# Baseline Modeling and Linear Regression

## 1. Objective

Establish a Dummy Regression baseline and compare it with Linear Regression using the same training cross-validation splits.

Combine preprocessing and each model in a single pipeline to prevent leakage during imputation, scaling, and encoding.

Evaluate performance using MAE, RMSE, and R². Keep the held-out test set reserved for final evaluation.

## 2. Import Required Libraries

Import the tools needed to load training data, reuse the preprocessing configuration, and evaluate Dummy Regression and Linear Regression through cross-validation.

In [23]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict


## 3. Load the Prepared Training Inputs

### 3.1 Load Training Predictors and Target

Load the saved predictors and original price target, preserving row indices. Read postcodes as text to maintain their categorical interpretation. Missing values remain for handling within the model pipeline.

In [2]:
project_root = Path.cwd().parent

X_train = pd.read_csv(
    project_root / "data/processed/X_train.csv",
    index_col="row_index",
    dtype={"Postcode": str}
)

y_train = pd.read_csv(
    project_root / "data/processed/y_train.csv",
    index_col="row_index"
)["Price"]

print("Predictors:", X_train.shape)
print("Target:", y_train.shape)

Predictors: (10864, 16)
Target: (10864,)


**Observation**

The training inputs contain 10,864 records and 16 predictors, with 10,864 corresponding target prices loaded separately. Missing predictor values remain for handling within the cross-validation pipeline. The held-out test data has not been loaded.

### 3.2 Verify Predictor–Target Alignment

Confirm that predictors and target have identical row indices in the same order, that record indices are unique, and that target prices contain no missing values.

In [3]:
print("Rows aligned:", X_train.index.equals(y_train.index))
print("Unique row indices:", X_train.index.is_unique)
print("Missing target values:", y_train.isna().sum())

Rows aligned: True
Unique row indices: True
Missing target values: 0


**Observation**

Predictors and target have identical row indices in the same order. Training row indices are unique, and no target prices are missing. The inputs are aligned for model training and cross-validation.

### 3.3 Load the Unfitted Preprocessor

Load the preprocessing configuration created in the previous notebook. It will be combined with each estimator and fitted within the cross-validation training folds.

In [4]:
preprocessor = joblib.load(
    project_root / "models/preprocessor_unfitted.joblib"
)

preprocessor

,transformers,"[('numerical', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


**Observation**

The saved unfitted preprocessor was loaded successfully. It contains the numerical and categorical preprocessing steps defined in the previous notebook. It will be fitted within each cross-validation training fold together with the estimator.

## 4. Define the Cross-Validation Setup

### 4.1 Create Reproducible Five-Fold Splits

Use shuffled five-fold cross-validation on the training data. Reuse the same splits for all baseline models to support fair comparison.

This random-split benchmark does not establish performance on future periods or guarantee separation of repeated properties. The held-out test set remains unused.

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_splits = list(cv.split(X_train, y_train))

print("Number of folds:", len(cv_splits))

Number of folds: 5


**Observation**

Five reproducible cross-validation splits were created from the training data. These same splits will be used for Dummy Regression and Linear Regression to ensure consistent comparison. The held-out test set remains unused.

### 4.2 Define Evaluation Metrics

Evaluate both models using MAE, RMSE, and R² on the same validation folds. Report MAE and RMSE in the original target units, with lower values indicating better performance. Higher R² indicates better performance relative to the evaluation-set mean baseline.

In [6]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

## 5. Establish a Dummy Regression Baseline

### 5.1 Define the Mean-Prediction Baseline

Use a DummyRegressor that predicts the training-fold mean price for every validation record. This provides a reference for evaluating the benefit of learning relationships between property features and prices.

Include the saved preprocessing configuration in the pipeline for workflow consistency. The dummy estimator itself ignores predictor values.

In [7]:
dummy_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DummyRegressor(strategy="mean"))
])

### 5.2 Evaluate the Dummy Baseline with Cross-Validation

Evaluate the mean-prediction pipeline using the five predefined training-validation splits. Fit preprocessing and the dummy estimator separately within each training fold, then calculate validation MAE, RMSE, and R². The held-out test set remains unused.

In [8]:
dummy_cv_results = cross_validate(
    dummy_pipeline,
    X_train,
    y_train,
    cv=cv_splits,
    scoring=scoring,
    error_score="raise"
)

print("Cross-validation completed.")

Cross-validation completed.


### 5.3 Inspect Dummy Baseline Scores by Fold

Display validation MAE, RMSE, and R² for each fold. Convert the negative error scores returned by scikit-learn into positive error values for interpretation.

In [9]:
dummy_fold_scores = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -dummy_cv_results["test_MAE"],
    "RMSE": -dummy_cv_results["test_RMSE"],
    "R2": dummy_cv_results["test_R2"]
})

dummy_fold_scores.round(3)

,Fold,MAE,RMSE,R2
0,1,449813.180,602944.729,-0.002
1,2,440660.109,582573.302,-0.004
2,3,467914.059,657238.204,-0.001
3,4,470902.335,662068.004,-0.000
4,5,482964.401,697625.442,-0.007


**Observation**

Across five validation folds, the mean-prediction baseline produced MAE values of approximately 440,660–482,964 and RMSE values of 582,573–697,625 in the original price units. R² scores were slightly negative, consistent with using training-fold means that differ from validation-fold means. These results establish a reference for assessing the predictive value of Linear Regression.

### 5.4 Summarize Dummy Baseline Performance

Summarize the mean and sample standard deviation of validation metrics across the five folds. The standard deviation describes variation between fold scores; it is not a confidence interval.

In [10]:
dummy_summary = (
    dummy_fold_scores[["MAE", "RMSE", "R2"]]
    .agg(["mean", "std"])
    .T
)

dummy_summary.round(3)

,mean,std
MAE,462450.817,17004.610
RMSE,640489.936,46836.318
R2,-0.003,0.003


**Observation**

The Dummy baseline achieved a mean cross-validation MAE of 462,450.817 and RMSE of 640,489.936, with fold-score standard deviations of 17,004.610 and 46,836.318, respectively. Mean R² was approximately −0.003. These results provide the reference against which Linear Regression will be compared using the same folds and metrics.

### 5.5 Save Dummy Baseline Results

Save the fold-level validation scores and their summary for reproducible reporting and comparison with subsequent models.

In [11]:
tables_dir = project_root / "results/tables"
tables_dir.mkdir(parents=True, exist_ok=True)

dummy_fold_scores.to_csv(
    tables_dir / "dummy_cv_fold_scores.csv", index=False
)

dummy_summary.to_csv(
    tables_dir / "dummy_cv_summary.csv", index_label="Metric"
)

## 6. Evaluate Linear Regression

### 6.1 Define the Linear Regression Pipeline

Combine a fresh copy of the preprocessing configuration with ordinary Linear Regression. During cross-validation, preprocessing and model parameters will be fitted using only each training fold. Evaluate the model using the same splits and metrics as the Dummy baseline.

In [12]:
linear_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LinearRegression())
])

### 6.2 Evaluate Linear Regression with Cross-Validation

Evaluate the Linear Regression pipeline using the same five splits and scoring metrics as the Dummy baseline. Fit preprocessing and model parameters within each training fold, then evaluate predictions on its validation fold. The held-out test set remains unused.

In [13]:
linear_cv_results = cross_validate(
    linear_pipeline,
    X_train,
    y_train,
    cv=cv_splits,
    scoring=scoring,
    return_train_score=True,
    error_score="raise"
)

print("Linear Regression cross-validation completed.")

Linear Regression cross-validation completed.


### 6.3 Inspect Linear Regression Scores by Fold

Display validation MAE, RMSE, and R² for each fold. Convert negative error scores to positive values for reporting while preserving the original R² scores.

In [14]:
linear_fold_scores = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -linear_cv_results["test_MAE"],
    "RMSE": -linear_cv_results["test_RMSE"],
    "R2": linear_cv_results["test_R2"]
})

linear_fold_scores.round(3)

,Fold,MAE,RMSE,R2
0,1,240909.076,352018.621,0.658
1,2,233040.336,346751.447,0.644
2,3,238881.533,383928.768,0.659
3,4,240766.864,460028.250,0.517
4,5,258379.089,434195.128,0.610


**Observation**

Linear Regression outperformed the Dummy baseline on all five validation folds, with lower MAE and RMSE and higher R². MAE ranged from approximately 233,040 to 258,379, while R² ranged from 0.517 to 0.659. Fold 4 had the highest RMSE despite a comparable MAE, suggesting a greater contribution from large prediction errors. Further error analysis is needed to investigate these cases.

### 6.4 Summarize Linear Regression Performance

Calculate the mean and sample standard deviation of validation MAE, RMSE, and R² across the five folds. These statistics summarize average performance and variation between fold scores.

In [15]:
linear_summary = (
    linear_fold_scores[["MAE", "RMSE", "R2"]]
    .agg(["mean", "std"])
    .T
)

linear_summary.round(3)

,mean,std
MAE,242395.379,9489.522
RMSE,395384.443,50154.976
R2,0.618,0.060


**Observation**

Linear Regression achieved a mean cross-validation MAE of 242,395.379, RMSE of 395,384.443, and R² of 0.618. Mean MAE was approximately 47.6% lower than the Dummy baseline. Fold-score standard deviations were 9,489.522 for MAE, 50,154.976 for RMSE, and 0.060 for R². The model provides a substantial improvement over mean-price prediction, although variation in RMSE warrants further error analysis.

### 6.5 Compare Training and Validation Performance

Compare mean training and validation scores across the same five folds. A substantial performance gap may indicate overfitting, while poor performance on both portions may suggest underfitting or limitations in the available features.

In [16]:
linear_train_valid = pd.DataFrame({
    "Training": [
        -linear_cv_results["train_MAE"].mean(),
        -linear_cv_results["train_RMSE"].mean(),
        linear_cv_results["train_R2"].mean()
    ],
    "Validation": [
        -linear_cv_results["test_MAE"].mean(),
        -linear_cv_results["test_RMSE"].mean(),
        linear_cv_results["test_R2"].mean()
    ]
}, index=["MAE", "RMSE", "R2"])

linear_train_valid.round(3)

,Training,Validation
MAE,223941.805,242395.379
RMSE,346765.413,395384.443
R2,0.708,0.618


**Observation**

Mean validation MAE was approximately 8.2% higher than training MAE, while validation RMSE was 14.0% higher. Mean R² decreased from 0.708 on training folds to 0.618 on validation folds. This indicates a generalization gap, but the aggregate scores alone do not establish severe overfitting. Further residual analysis and comparison with regularized models are needed.

### 6.6 Save Linear Regression Results

Save fold-level validation scores, their summary, and the training–validation comparison for reproducible reporting and later model comparisons.

In [18]:
linear_fold_scores.to_csv(
    tables_dir / "linear_cv_fold_scores.csv", index=False
)

linear_summary.to_csv(
    tables_dir / "linear_cv_summary.csv", index_label="Metric"
)

linear_train_valid.to_csv(
    tables_dir / "linear_train_validation_summary.csv",
    index_label="Metric"
)

## 7. Compare Baseline Models

### 7.1 Compare Mean Cross-Validation Scores

Compare Dummy Regression and Linear Regression using mean validation metrics from the same five folds. Lower MAE and RMSE and higher R² indicate better predictive performance.

In [19]:
model_comparison = pd.DataFrame({
    "Dummy": dummy_summary["mean"],
    "Linear Regression": linear_summary["mean"]
}).T

model_comparison.round(3)

,MAE,RMSE,R2
Dummy,462450.817,640489.936,-0.003
Linear Regression,242395.379,395384.443,0.618


**Observation**

Linear Regression outperformed the Dummy baseline on all three mean cross-validation metrics. MAE decreased by approximately 47.6% and RMSE by 38.3%, while mean R² increased from −0.003 to 0.618. These results demonstrate predictive value beyond constant mean-price prediction under the current validation design. The held-out test set remains unevaluated.

### 7.2 Save the Baseline Comparison

Export the mean cross-validation scores for Dummy Regression and Linear Regression. These results provide a reference for subsequent model comparisons.

In [21]:
model_comparison.to_csv(
    tables_dir / "baseline_model_comparison.csv",
    index_label="Model"
)

## 8. Analyze Out-of-Fold Predictions and Residuals

### 8.1 Generate Out-of-Fold Predictions

Generate one prediction per training record using the same five cross-validation splits. Each record is predicted by a pipeline fitted without that record. These predictions support residual analysis while keeping the held-out test set unused.

In [24]:
linear_oof_predictions = cross_val_predict(
    linear_pipeline,
    X_train,
    y_train,
    cv=cv_splits,
    method="predict"
)

print("OOF prediction shape:", linear_oof_predictions.shape)

OOF prediction shape: (10864,)


**Observation**

Out-of-fold predictions were generated for all 10,864 training records using the predefined five-fold splits. Each record was predicted by a pipeline fitted without that record, with predictions returned in the original input order. The held-out test set remains unused.

### 8.2 Calculate Out-of-Fold Residuals

Compare each original training price with its out-of-fold prediction. Calculate residuals as actual minus predicted price, and absolute errors to measure error magnitude regardless of direction.

In [25]:
linear_oof_results = pd.DataFrame({
    "Actual_Price": y_train,
    "Predicted_Price": linear_oof_predictions
})

linear_oof_results["Residual"] = (
    linear_oof_results["Actual_Price"]
    - linear_oof_results["Predicted_Price"]
)

linear_oof_results["Absolute_Error"] = (
    linear_oof_results["Residual"].abs()
)

linear_oof_results.head().round(2)

,Actual_Price,Predicted_Price,Residual,Absolute_Error
row_index,,,,
12796,1480000.0,1548772.21,-68772.21,68772.21
9642,1530000.0,1197999.83,332000.17,332000.17
3207,690000.0,875543.59,-185543.59,185543.59
1698,570000.0,847117.04,-277117.04,277117.04
761,1041000.0,1480322.06,-439322.06,439322.06


**Observation**

The displayed records contain both positive and negative out-of-fold residuals, indicating underprediction and overprediction. Absolute errors measure the magnitude of each discrepancy without its direction. These five records illustrate the calculations but are insufficient to establish overall prediction bias.

### 8.3 Summarize Out-of-Fold Residuals

Summarize residuals and absolute errors across all training records. Examine their centre, spread, and extremes before inspecting residual plots and individual high-error cases.

In [26]:
linear_oof_results[
    ["Residual", "Absolute_Error"]
].describe().round(2)

,Residual,Absolute_Error
count,10864.00,10864.00
mean,-571.26,242393.91
std,397935.61,315583.36
min,-13055543.13,37.88
25%,-191537.59,76944.80
50%,-22120.33,171330.00
75%,147464.55,315350.07
max,8492988.11,13055543.13


**Observation**

The mean OOF residual was −571.26, but this near-zero signed average masks substantial individual errors. Mean absolute error was 242,393.91, compared with a median of 171,330.00, indicating that large errors raise the average. The largest overprediction was approximately 13.06 million, while the largest underprediction was 8.49 million. These extreme cases require record-level inspection before drawing conclusions about their causes.

### 8.4 Inspect the Largest Out-of-Fold Errors

Inspect the ten records with the largest absolute prediction errors alongside their selected property features. This review investigates potential explanations without automatically removing records or changing values.

In [27]:
largest_errors = (
    linear_oof_results
    .join(X_train[[
        "Suburb", "Type", "Rooms", "Distance",
        "Landsize", "BuildingArea", "YearBuilt"
    ]])
    .nlargest(10, "Absolute_Error")
)

largest_errors.round(2)

,Actual_Price,Predicted_Price,Residual,Absolute_Error,Suburb,Type,Rooms,Distance,Landsize,BuildingArea,YearBuilt
row_index,,,,,,,,,,,
13245,1355000.0,14410543.13,-13055543.13,13055543.13,New Gisborne,h,5,48.1,44500.0,44515.0,NaN
12094,9000000.0,507011.89,8492988.11,8492988.11,Mulgrave,h,3,18.8,744.0,117.0,1960.0
9575,7650000.0,2436101.60,5213898.40,5213898.40,Hawthorn,h,4,5.3,1690.0,284.0,1863.0
7692,8000000.0,3189458.18,4810541.82,4810541.82,Canterbury,h,5,9.0,2079.0,464.3,1880.0
6372,6250000.0,2268850.18,3981149.82,3981149.82,Toorak,h,3,4.6,564.0,342.0,2000.0
12557,6400000.0,2513003.09,3886996.91,3886996.91,Middle Park,h,5,3.0,553.0,308.0,1920.0
5631,5700000.0,1917583.69,3782416.31,3782416.31,South Yarra,h,4,3.3,292.0,272.0,1880.0
1484,1280000.0,4978418.32,-3698418.32,3698418.32,Bulleen,h,4,11.8,732.0,6791.0,NaN
3114,5100000.0,1792382.56,3307617.44,3307617.44,Hawthorn,h,3,4.6,1038.0,NaN,NaN
